In [5]:
import os
if os.name == 'nt': # Check if windows
    os.add_dll_directory(r'C:\Program Files\SuperTuxKart 1.5')

import multiprocessing as mp
import numpy as np

import torch
import torch.optim as optim
import torch.nn.functional as F
from actor import ActorNetwork
from critic import CriticNetwork
from env_worker import SingleInstance

In [6]:
# --- STEP 1: THE PPO UPDATE FUNCTION ---

# Instantiate models with an input dimension of 44
actor_net = ActorNetwork(state_dim=44)
critic_net = CriticNetwork(state_dim=44)

# Set to evaluation mode for simulation
actor_net.eval()
critic_net.eval()

# Initialize Optimizer for both networks
optimizer = optim.Adam([
	{'params': actor_net.parameters(), 'lr': 1e-4},
	{'params': critic_net.parameters(), 'lr': 1e-4}
])

def update_ppo(buffer, epochs=4, gamma=0.99, clip_epsilon=0.1):
	# Extract Data from Buffer
	states = torch.FloatTensor(np.array([t['state'] for t in buffer]))
	
	# Reconstruct the original Actor format of [Steering, Acceleration].
	actions = torch.FloatTensor([[t['action'][0], t['action'][1], t['action'][2]] for t in buffer])
	
	rewards = [t['reward'] for t in buffer]
	values = torch.FloatTensor([t['value'] for t in buffer]).unsqueeze(1)
	old_log_probs = torch.FloatTensor([t['log_prob'] for t in buffer]).unsqueeze(1)
	
	# Calculate Discounted Returns
	returns = []
	discounted_reward = 0
	for reward in reversed(rewards):
		discounted_reward = reward + (gamma * discounted_reward)
		returns.insert(0, discounted_reward)
	returns = torch.FloatTensor(returns).unsqueeze(1)
	
	# Calculate Advantages
	advantages = returns - values
	# Added 1e-8 to prevent division by zero
	advantages = (advantages - advantages.mean()) / (advantages.std(unbiased=False) + 1e-8)
	
	# Switch models back to train mode
	actor_net.train()
	critic_net.train()
	
	# PPO Update Loop
	for _ in range(epochs):
		# Check for corrupted engine data
		if torch.isnan(states).any():
			print("NaN detected in states buffer! Skipping PPO update.")
			break
			
		action_dists = actor_net(states)
		new_values = critic_net(states)
		
		new_log_probs = action_dists.log_prob(actions).sum(dim=-1, keepdim=True)
		entropy = action_dists.entropy().sum(dim=-1, keepdim=True).mean()
		
		ratio = torch.exp(new_log_probs - old_log_probs)
		
		surr1 = ratio * advantages
		surr2 = torch.clamp(ratio, 1.0 - clip_epsilon, 1.0 + clip_epsilon) * advantages
		
		actor_loss = -torch.min(surr1, surr2).mean()
		critic_loss = F.mse_loss(new_values, returns)
		
		loss = actor_loss + 0.5 * critic_loss - 0.05 * entropy
		
		optimizer.zero_grad()
		loss.backward()
		
		# Gradient Clipping to prevent exploding weights
		torch.nn.utils.clip_grad_norm_(actor_net.parameters(), max_norm=0.5)
		torch.nn.utils.clip_grad_norm_(critic_net.parameters(), max_norm=0.5)
		
		optimizer.step()
		
	actor_net.eval()
	critic_net.eval()

In [7]:
def main():
	try:
		PATIENCE_LIMIT = 50
		best_reward = float('-inf')
		patience_counter = 0

		for episode in range(100):
			buffer = []
			total_episode_reward = 0.0
			ProcessList = []
			ConList = []
			for i in range(5):
				ParentCon,ChildCon = mp.Pipe()
				process = mp.Process(target=SingleInstance,args=(i,ChildCon))
				ProcessList.append(process)
				ConList.append(ParentCon)
				process.start()
			BatchStates = []
			BatchDones = []

			for con in ConList:
					np_obs,reward,RaceDone = con.recv()
					BatchStates.append(np_obs)
					BatchDones.append(RaceDone)

			for step in range(1000):

				# --- Phase 2 Brain Injection ---
				state_tensor = torch.FloatTensor(np.array(BatchStates))
				
				# Sanity Check for incoming engine observations
				if torch.isnan(state_tensor).any():
					print("NaN detected in engine observations! Terminating episode.")
					break
					
				with torch.no_grad():
					action_dist = actor_net(state_tensor)
					sampled_action = action_dist.sample()
					state_value = critic_net(state_tensor)

					BatchLogProbs = action_dist.log_prob(sampled_action).sum(dim=-1)
				
				MemoryActions = []

				for i,con in enumerate(ConList):
					steer_val = torch.clamp(sampled_action[i, 0], min=-1.0, max=1.0).item()
					accel_val = torch.clamp(sampled_action[i, 1], min=0.0, max=1.0).item()
					BrakeVal = torch.clamp(sampled_action[i,2],min=0.0,max=1.0).item()
					MemoryActions.append((steer_val,accel_val,BrakeVal))

					if BatchDones[i]:
						con.send('TERMINATE')
					else:
						con.send((steer_val,accel_val,BrakeVal))

				NextStates = []
				PreviousRewards = []
				PreviousDones = []

				for con in ConList:
						np_obs,reward,RaceDone = con.recv()
						NextStates.append(np_obs)
						PreviousRewards.append(reward)
						PreviousDones.append(RaceDone)

				for i in range(len(ConList)):
					transition = {
						"state": BatchStates[i],
						"action": np.array([MemoryActions[i][0], MemoryActions[i][1], MemoryActions[i][2], 0.0, 0.0], dtype=np.float32), # Steer, Accel, Brake, Drift?, Nitro?
						"reward": PreviousRewards[i],
						"value": state_value[i].item(),
						"log_prob": BatchLogProbs[i].item(),
						"done": PreviousDones[i]
					}
					# Buffer Appending
					buffer.append(transition)
				
				total_episode_reward += sum(PreviousRewards)

				BatchStates = NextStates
				BatchDones = PreviousDones
					
			# --- END OF EPISODE TRIGGER ---
			print(f"Episode: {episode + 1}/100 | Total Reward: {(total_episode_reward/max(len(buffer),1)):.2f} | Buffer Size: {len(buffer)}")

			# Trigger the Brain Transplant!
			update_ppo(buffer)

			mean_reward = total_episode_reward / max(len(buffer), 1)
			if mean_reward > best_reward:
				best_reward = mean_reward
				patience_counter = 0
				torch.save(actor_net.state_dict(), "best_actor.pth")
				torch.save(critic_net.state_dict(), "best_critic.pth")
				print(f"  New best reward: {best_reward:.4f} - models saved.")
			else:
				patience_counter += 1
				print(f"  No improvement. Patience: {patience_counter}/{PATIENCE_LIMIT}")

			if patience_counter >= PATIENCE_LIMIT:
				print(f"\nEarly stopping triggered after {episode + 1} episodes.")
				break

			for con in ConList:
				con.send('TERMINATE')

			for process in ProcessList:
				process.join()
	
	finally:
		for con in ConList:
			con.send('TERMINATE')

		for process in ProcessList:
			process.join()

In [8]:
if __name__ == '__main__':
	main()

..:: Antarctica Rendering Engine 2.0 ::..
..:: Antarctica Rendering Engine 2.0 ::..
..:: Antarctica Rendering Engine 2.0 ::..
..:: Antarctica Rendering Engine 2.0 ::..
..:: Antarctica Rendering Engine 2.0 ::..
Episode: 1/100 | Total Reward: -3.41 | Buffer Size: 5000
  New best reward: -3.4146 - models saved.
..:: Antarctica Rendering Engine 2.0 ::..
..:: Antarctica Rendering Engine 2.0 ::..
..:: Antarctica Rendering Engine 2.0 ::..
..:: Antarctica Rendering Engine 2.0 ::..
..:: Antarctica Rendering Engine 2.0 ::..
Episode: 2/100 | Total Reward: 11.97 | Buffer Size: 5000
  New best reward: 11.9730 - models saved.
..:: Antarctica Rendering Engine 2.0 ::..
..:: Antarctica Rendering Engine 2.0 ::..
..:: Antarctica Rendering Engine 2.0 ::..
..:: Antarctica Rendering Engine 2.0 ::..
..:: Antarctica Rendering Engine 2.0 ::..
Episode: 3/100 | Total Reward: 13.33 | Buffer Size: 5000
  New best reward: 13.3306 - models saved.
..:: Antarctica Rendering Engine 2.0 ::..
..:: Antarctica Rendering En